# Course 2 lab — Evidence-aware agent risk modeling

**Scenario:** Northstar is considering three procurement-agent capabilities: catalogue search, purchase-order creation, and vendor payment. The governance team must classify autonomy, model operational and adversarial scenarios, map blast radius, select control profiles, and decide what evidence is required before residual risk can be lowered.

## Outcomes

You will classify observable authority, retain a multidimensional risk vector, separate failure from attack, analyze exact reachable assets with NetworkX, compare a weighted-score baseline with explicit decision rules, and accept control credit only when current scenario-specific evidence supports it.

**Boundary:** this is an organization-specific teaching policy, not a universal risk calculator, legal classification, or production certification.

![Agent risk dimensions](assets/01-agent-risk-dimensions.svg)

The core workflow is:

```text
capability -> observable autonomy -> scenario -> dimensions + evidence
           -> inherent tier -> controls -> tested claims -> residual tier -> decision
```

A tier routes governance work. It does not erase the scenario, assumptions, raw dimensions, or uncertainty.

## 1. Offline setup

The notebook imports the tested `lab.py`. Pydantic defines contracts and NetworkX makes downstream reachability inspectable. No model API or credential is used.

In [ ]:
from datetime import date
from pathlib import Path
from pprint import pprint
import sys

topic_dir = Path.cwd()
if not (topic_dir / 'lab.py').exists():
    topic_dir = (Path.cwd() / 'curriculum/beginner/02-agent-risk-modeling-and-autonomy-classification').resolve()
if str(topic_dir) not in sys.path:
    sys.path.insert(0, str(topic_dir))

from lab import (
    AutonomyLevel, AutonomySignals, ControlEvidence, ControlType, EvidenceStatus,
    MitigationClaim, OrdinalLevel, RiskTier, assess, blast_radius, classify_autonomy,
    classify_risk, demo_capabilities, demo_scenarios, evaluation_report, procurement_graph,
)
capabilities = demo_capabilities()
scenarios = {item.scenario_id: item for item in demo_scenarios()}
print('Loaded:', topic_dir / 'lab.py')

## 2. Baseline: why one weighted decimal is misleading

A common spreadsheet multiplies or averages ordinal labels and returns a precise-looking number. Two very different situations can receive the same result, while the number hides catastrophic severity, poor evidence, or a wide blast radius. This baseline is included only to expose that failure.

In [ ]:
def opaque_average(values):
    return round(sum(values) / len(values), 2)

case_a = [4, 1, 1, 4, 2]  # catastrophic and irreversible, but described as unlikely
case_b = [2, 3, 3, 2, 2]  # more frequent and detectable, lower consequence
print('Case A:', opaque_average(case_a), '| Case B:', opaque_average(case_b))
assert opaque_average(case_a) == opaque_average(case_b)
print('The same average does not imply the same treatment.')

## 3. Classify autonomy from observable authority

Framework names do not determine autonomy. The relevant questions are whether the system prepares or executes state changes, whether every write is preapproved, whether it selects tools/plans steps, whether it delegates, and who owns termination.

In [ ]:
profiles = {
    'recommendation': AutonomySignals(),
    'human_executes': AutonomySignals(prepares_state_change=True, selects_tools_or_plans_steps=True),
    'bounded_write': AutonomySignals(prepares_state_change=True, executes_state_change=True, every_state_change_preapproved=True),
    'delegating_executor': AutonomySignals(prepares_state_change=True, executes_state_change=True, every_state_change_preapproved=False, can_delegate=True, determines_completion=True),
}
for name, signals in profiles.items():
    result = classify_autonomy(signals)
    print(name, '->', result.level.name, result.reasons, result.review_triggers)

Autonomy is only one input. A high-autonomy sandbox can remain low risk, while a bounded payment capability can require critical controls because impact and irreversibility differ.

In [ ]:
for capability in capabilities.values():
    pprint(capability.model_dump(mode='json'))

## 4. Keep operational failure and adversarial misuse distinct

FMEA-style reasoning is useful for accidental or operational failures. OWASP Agentic Security, MITRE ATLAS, and NIST AI 100-2 help structure motivated-adversary scenarios. They can share controls, but mixing their likelihood assumptions and evidence produces weak decisions.

In [ ]:
for scenario in scenarios.values():
    print(scenario.scenario_id, scenario.kind.value, '-', scenario.title)
    print('  dimensions:', scenario.dimensions.model_dump(mode='json'))
    print('  evidence:', scenario.evidence_status.name, scenario.evidence_refs or scenario.assumptions)

## 5. Measure blast radius with exact graph facts

A normalized `0.73` blast score is difficult to defend. The lab reports which assets are reachable and writable, which severe assets are exposed, how many trust-zone crossings occur, and whether authority is delegated.

In [ ]:
broad = blast_radius(procurement_graph(include_finance=True))
constrained = blast_radius(procurement_graph(include_finance=False))
print('Broad graph:')
pprint(broad)
print('After removing direct payment reachability:')
pprint(constrained)
assert 'payment_service' in broad.severe_assets
assert 'payment_service' not in constrained.reachable_assets

The graph experiment demonstrates a treatment that changes architecture rather than merely lowering a spreadsheet score. Removing a capability reduces the set of reachable severe assets and creates evidence reviewers can inspect.

## 6. Apply explicit risk gates

The teaching policy retains severity, likelihood, detectability difficulty, irreversibility, and scope. Its rules deliberately prioritize severe irreversible or broad harm rather than averaging it away. Your organization must version and approve its own thresholds.

In [ ]:
for scenario in scenarios.values():
    capability = capabilities[scenario.capability_id]
    tier, reasons = classify_risk(scenario.dimensions, capability.autonomy)
    print(scenario.scenario_id, '->', tier.name, reasons)

assert classify_risk(scenarios['ADV-001'].dimensions, capabilities['payment'].autonomy)[0] is RiskTier.CRITICAL

## 7. Do not grant control credit without evidence

A control name in a risk register is not proof. A mitigation claim identifies one scenario and one dimension. It applies only when tested or observed evidence covers that scenario, is current, and matches the baseline being reduced.

In [ ]:
retry_scenario = scenarios['OPS-001']
claim = MitigationClaim(
    scenario_id='OPS-001', control_id='idempotency', dimension='likelihood',
    from_level=OrdinalLevel.MODERATE, to_level=OrdinalLevel.LOW,
)
documented_only = ControlEvidence(
    evidence_id='e-doc', control_id='idempotency', scenario_ids=frozenset({'OPS-001'}),
    control_type=ControlType.PREVENT, status=EvidenceStatus.DOCUMENTED, evidence_ref='design-doc-12',
)
no_credit = assess(retry_scenario, capabilities['purchase_order'], claims=[claim], evidence=[documented_only])
print('Documented only:', no_credit.residual_tier.name, no_credit.rejected_claims)
assert no_credit.residual_dimensions == no_credit.inherent_dimensions

In [ ]:
tested = ControlEvidence(
    evidence_id='e-test', control_id='idempotency', scenario_ids=frozenset({'OPS-001'}),
    control_type=ControlType.PREVENT, status=EvidenceStatus.TESTED,
    evidence_ref='tests/test_module01_governance.py::test_idempotent_retry_does_not_duplicate_effect',
    valid_through=date(2026, 12, 31),
)
with_credit = assess(retry_scenario, capabilities['purchase_order'], claims=[claim], evidence=[tested])
print('Tested evidence:', with_credit.residual_tier.name, with_credit.applied_claims)
print('Disposition:', with_credit.disposition)
print('Required controls:', with_credit.required_controls)
assert with_credit.residual_dimensions.likelihood is OrdinalLevel.LOW

### Failure injection: stale evidence

A once-passing test can become stale after code, policy, model, tool, or environment changes. Expired evidence must not continue reducing residual risk.

In [ ]:
expired = tested.model_copy(update={'evidence_id': 'e-expired', 'valid_through': date(2026, 1, 1)})
stale = assess(retry_scenario, capabilities['purchase_order'], claims=[claim], evidence=[expired])
print(stale.rejected_claims, stale.disposition)
assert stale.rejected_claims == ('idempotency:likelihood:evidence_expired',)

## 8. Evaluate the classifier on a labelled fixture

The metric population is three versioned teaching cases: duplicate PO, adversarial payment redirection, and stale catalogue recommendation. Accuracy means correct tier classifications divided by those three cases. It says nothing about unlabelled risks, control operation, or production safety.

In [ ]:
report = evaluation_report()
pprint(report)
assert report == {'correct': 3, 'cases': 3, 'classification_accuracy': 1.0}

## 9. Methods and tool landscape

| Need | Method/tool | Strength | Limitation |
|---|---|---|---|
| Enterprise risk lifecycle | NIST AI RMF, ISO/IEC 23894 | Connects context, measurement, treatment, ownership | Does not supply one universal agent score |
| Operational failure | FMEA, bow-tie analysis, STPA where safety constraints matter | Makes causes, effects, prevention, detection, recovery explicit | Ordinal RPNs can hide different risk shapes |
| Adversarial threat | OWASP Agentic Top 10, MITRE ATLAS, NIST AI 100-2 | Current threat language and attack paths | A taxonomy is not evidence that a control works |
| Capability and dependency graph | NetworkX, architecture inventories, attack graphs | Exact reachability and trust-zone analysis | Graph quality depends on inventory freshness |
| Structured contracts | Pydantic, JSON Schema | Reproducible registers and validation | Typed input is not correct or authorized evidence |
| Adversarial testing | PyRIT and custom deterministic harnesses | Turns threats into executable tests | Scores require validated oracles and representative targets |
| Candidate discovery | LLM structured extraction | Helps brainstorm scenarios from architecture text | Candidates require human/security validation; the model cannot accept risk |

Use spreadsheets for workshops, code for repeatability, graph tools for reachability, and red-team harnesses for exploitability. No single library replaces accountable risk ownership.

## 10. Production upgrade path

- version the methodology, thresholds, taxonomies, graph snapshot, and every assessment;
- derive capabilities and access paths from authoritative inventories rather than self-report;
- separate internal tier, regulatory applicability, and exception decisions;
- store assumptions, owners, evidence IDs, review dates, change triggers, and treatment state;
- require independent review for severe, cross-tenant, privileged, financial, or safety effects;
- connect release gates to tests without letting a numeric score auto-authorize deployment;
- recertify after model, prompt, tool, permission, data, workflow, policy, or dependency changes; and
- monitor leading indicators and verified business outcomes, not only blocked attempts.

## 11. Exercises

1. **Implement:** add an external supplier-email capability and two scenarios, one operational and one adversarial.
2. **Diagnose:** create two dimension vectors with the same weighted average but different required treatment.
3. **Graph experiment:** give the research sub-agent write access to the vendor database; report exact delta in reachability, writable assets, and trust-zone crossings.
4. **Evidence:** design a test that justifies one mitigation claim. State population, oracle, failure cases, validity period, and invalidation triggers.
5. **Architecture:** compare FMEA, bow-tie, STPA, OWASP, and ATLAS for a healthcare scheduling agent; explain which methods complement rather than replace one another.
6. **Governance judgment:** decide whether the payment scenario should be redesigned, restricted, accepted by exception, or prohibited—and identify the accountable decision owner.

## 12. Checkpoint

1. Why can two capabilities at the same autonomy level require different tiers?
2. Why is an FMEA RPN an ordering aid rather than an objective probability?
3. What exact evidence would justify lowering likelihood after adding idempotency?
4. When should architecture reduction be preferred over adding another control?
5. Why must operational failures and adversarial threats remain distinguishable?

**Exit criterion:** defend a tier using visible scenario facts, reject unsupported residual-risk reduction, and show how a capability or graph change alters the decision.

## Primary and official references

- NIST AI RMF and revision status: https://www.nist.gov/itl/ai-risk-management-framework
- NIST AI RMF Generative AI Profile: https://www.nist.gov/publications/artificial-intelligence-risk-management-framework-generative-artificial-intelligence
- ISO/IEC 23894:2023: https://www.iso.org/standard/77304.html
- NIST AI 100-2e2025 adversarial ML taxonomy: https://csrc.nist.gov/pubs/ai/100/2/e2025/final
- NIST AI 800-5 agent-security response analysis: https://www.nist.gov/publications/summary-analysis-responses-request-information-regarding-security-considerations-ai
- OWASP Top 10 for Agentic Applications 2026: https://genai.owasp.org/resource/owasp-top-10-for-agentic-applications-for-2026/
- MITRE ATLAS: https://atlas.mitre.org/
- NetworkX: https://networkx.org/documentation/stable/
- PyRIT: https://azure.github.io/PyRIT/